In [81]:
import pandas as pd

In [82]:
df=pd.read_csv("IMDB Dataset.csv")

In [83]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [84]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [85]:
df.drop_duplicates(inplace=True)

In [86]:
df.shape

(49582, 2)

## Pre-Processing

### 1.Convert to Lowercase

In [87]:
df["review"]=df["review"].str.lower()

In [88]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


### 2.Remove URLs

In [89]:
import re
def remove_urls(text):
    text=re.sub(r"http\S+","",text)
    return text

df["review"]=df["review"].apply(remove_urls)

In [90]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


### 3.Remove Punctuations

In [91]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]","",text)
    return text

df["review"]=df["review"].apply(remove_punctuations)

In [92]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 4.Removing HTML

In [93]:
def remove_html(text):
    text=re.sub(r"<.*?>","",text)
    return text

df["review"]=df["review"].apply(remove_html)

### 5.Removing StopWords

In [94]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\prash\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\prash\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\prash\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [95]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [96]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text=text.replace(word,"")

    return text
    
df["review"]=df["review"].apply(remove_stopwords)

In [97]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6.Stemming

In [98]:
from nltk.stem import PorterStemmer

def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]
    
    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)
    
df["review"]=df["review"].apply(stemming)

In [99]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7.Encoding

In [100]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])

In [101]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


### 8.Vectorization

In [103]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer=TfidfVectorizer(max_features=5000)
X=vectorizer.fit_transform(df["review"])

In [104]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057169 stored elements and shape (49582, 5000)>

In [107]:
y=df["sentiment"]

### Dataset && DataLoaders

In [138]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [139]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset,DataLoader

In [140]:
X_train=X_train.toarray()
X_test=X_test.toarray()

In [141]:
train_dataset=TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_dataset=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [142]:
train_loader=DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=64,shuffle=True)

## Build RNN

In [143]:
# Build our Model 

class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super(RNN,self).__init__()

        self.hidden_size=hidden_size
        self.num_layers=num_layers

        # RNN Layer
        self.rnn=nn.RNN(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True
        )

        # Fully connected Layer
        self.fc=nn.Linear(hidden_size,1)

    def forward(self,x):

        h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)

        out,_ = self.rnn(x,h0)

        out=self.fc(out[:,-1,:])
        return out

In [144]:
input_size=X_train.shape[1]

model=RNN(input_size)

criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

In [145]:
epochs=10
model.train()

for epoch in range(epochs):

    for Xb,yb in train_loader:
        optimizer.zero_grad()

        Xb=Xb.unsqueeze(1)
        outputs=model(Xb)
        outputs=torch.sigmoid(outputs.squeeze())
        loss=criterion(outputs,yb)
        loss.backward()

        optimizer.step()

    print(f"epoch = {epoch+1} ,loss = {loss.item()}")

epoch = 1 ,loss = 0.34468868374824524
epoch = 2 ,loss = 0.2521979510784149
epoch = 3 ,loss = 0.22993282973766327
epoch = 4 ,loss = 0.2963559031486511
epoch = 5 ,loss = 0.19840872287750244
epoch = 6 ,loss = 0.16416791081428528
epoch = 7 ,loss = 0.1526375710964203
epoch = 8 ,loss = 0.1719214767217636
epoch = 9 ,loss = 0.20661011338233948
epoch = 10 ,loss = 0.11088511347770691


In [149]:
# Evaluation 

model.eval()

with torch.no_grad():
    total = 0
    correct = 0

    for Xb,yb in test_loader:
        Xb=Xb.unsqueeze(1)

        outputs=model(Xb)
        predicted=(torch.sigmoid(outputs.squeeze()) > 0.5).float()

        total += yb.size(0)
        correct += (predicted == yb).sum().item()

print("Accuracy : ",correct/total)

Accuracy :  0.8570132096400122
